# SNS Studio Drive Job Worker

This notebook polls job folders created by SNS Studio, calls a separately installed model adapter, uploads its outputs to the selected Drive folder, and writes a result manifest that SNS Studio can import. No model provider is bundled because the project has not selected one.

Add your model adapter as `sns_studio_model_adapter.py` in the Colab runtime. It must expose `generate(job: dict, output_dir: str) -> list[str]` and return generated local file paths. The adapter owns model-specific setup and credentials.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaIoBaseUpload, MediaFileUpload
import io, json, mimetypes, os, tempfile, time

credentials, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
drive = build('drive', 'v3', credentials=credentials)
ROOT_FOLDER_ID = 'PASTE_THE_SELECTED_SNS_STUDIO_FOLDER_ID_HERE'
POLL_SECONDS = 60

In [ ]:
def download_json(file_id):
    request = drive.files().get_media(fileId=file_id)
    output = io.BytesIO()
    downloader = MediaIoBaseDownload(output, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    return json.loads(output.getvalue().decode('utf-8'))

def upload_json(parent_id, name, value):
    body = {'name': name, 'mimeType': 'application/json', 'parents': [parent_id]}
    media = MediaIoBaseUpload(io.BytesIO(json.dumps(value, ensure_ascii=False).encode('utf-8')), mimetype='application/json')
    return drive.files().create(body=body, media_body=media, fields='id,name').execute()

def find_job_folders():
    response = drive.files().list(q=f"'{ROOT_FOLDER_ID}' in parents and mimeType='application/vnd.google-apps.folder' and name contains 'SNSStudio_job_' and trashed=false", fields='files(id,name)', pageSize=1000).execute()
    return response.get('files', [])

def result_exists(job_id):
    name = f'sns-studio-result-{job_id}.json'
    response = drive.files().list(q=f"'{ROOT_FOLDER_ID}' in parents and name='{name}' and trashed=false", fields='files(id)', pageSize=1).execute()
    return bool(response.get('files'))

def write_result(job, output_paths):
    uploaded = []
    for path in output_paths:
        mime_type = mimetypes.guess_type(path)[0] or 'application/octet-stream'
        file = drive.files().create(body={'name': os.path.basename(path), 'parents': [ROOT_FOLDER_ID]}, media_body=MediaFileUpload(path, mimetype=mime_type, resumable=True), fields='id,name,mimeType').execute()
        uploaded.append({'fileId': file['id'], 'name': file['name'], 'mimeType': file.get('mimeType', mime_type)})
    result = {'jobId': job['jobId'], 'status': 'COMPLETE', 'outputs': uploaded, 'completedAt': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
    return upload_json(ROOT_FOLDER_ID, f"sns-studio-result-{job['jobId']}.json", result)

def write_failure(job, error):
    result = {'jobId': job['jobId'], 'status': 'FAILED', 'errorCode': 'MODEL_ADAPTER_FAILED', 'errorMessage': str(error)[:400]}
    return upload_json(ROOT_FOLDER_ID, f"sns-studio-result-{job['jobId']}.json", result)

In [ ]:
def run_one_job(folder):
    files = drive.files().list(q=f"'{folder['id']}' in parents and name='job.json' and trashed=false", fields='files(id)', pageSize=1).execute().get('files', [])
    if not files:
        return False
    job = download_json(files[0]['id'])
    if result_exists(job['jobId']):
        return False
    with tempfile.TemporaryDirectory(prefix='sns-studio-') as output_dir:
        try:
            from sns_studio_model_adapter import generate
            output_paths = generate(job, output_dir)
            if not output_paths:
                raise RuntimeError('The model adapter returned no output files.')
            write_result(job, output_paths)
        except Exception as error:
            write_failure(job, error)
    return True

def poll_once():
    processed = 0
    for folder in find_job_folders():
        try:
            processed += int(run_one_job(folder))
        except Exception as error:
            print(f"Skipped {folder.get('name')}: {error}")
    print(f'Processed {processed} job(s)')
    return processed

poll_once()

Run `poll_once()` manually for a one-shot run, or use this cell while the Colab runtime remains active to poll periodically. SNS Studio imports result manifests from Settings → Generation Jobs → Sync results. Colab runtimes can stop at any time; use the button to run this notebook again.